# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, following the Croissant schema.

### Dataset Source
The dataset Croissant schema is provided via:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if not present
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and explore key information from the Croissant metadata object.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n\nDescription: {metadata.description}")
print(f"\nVersion: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Query the available record sets and their fields using their `@id` references as described in the schema.

In [ ]:
# List available record sets and their field information using @id

record_sets = [record_set for record_set in dataset.record_sets]
print(f"Number of record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"@id: {rs.id}")
    print("Fields (by @id and name):")
    for field in rs.fields:
        print(f"  - @id: {field.id} | name: {field.name}")
    print("-"*40)

## 3. Data Extraction
Extract data from a selected record set into a DataFrame. When referencing any record set or field, we use its `@id`.

> For this dataset, the most comprehensive record set typically contains protein-level mass spectrometry results (often named `results` or similar, check exact name and id above).

In [ ]:
# Build dataframes for all available record sets by their @id

dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded DataFrame for: {rs.id} ({len(df)} rows)")

# For illustration, select the largest record set
main_record_set = max(record_sets, key=lambda rs: len(dataframes[rs.id]))
main_rs_id = main_record_set.id
print(f"\nSelected main record set for EDA: {main_rs_id}\n")
print("Columns in the main record set:")
pprint(dataframes[main_rs_id].columns.tolist())

dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

We demonstrate common EDA operations: filtering, normalization, grouping, and basic statistics on selected numeric fields.

We use the field `@id` to select fields/columns. Make sure to cross-check the available fields above.

In [ ]:
# For demonstration, find a numeric field (e.g., molecular weight or abundance value)

# Detect likely numeric columns by dtype
numeric_cols = dataframes[main_rs_id].select_dtypes(include='number').columns.tolist()
if not numeric_cols:
    # Sometimes numeric fields are read as object if numbers are strings.
    for col in dataframes[main_rs_id].columns:
        try:
            dataframes[main_rs_id][col] = pd.to_numeric(dataframes[main_rs_id][col], errors='ignore')
        except Exception:
            pass
    numeric_cols = dataframes[main_rs_id].select_dtypes(include='number').columns.tolist()

if numeric_cols:
    numeric_field = numeric_cols[0]  # Select the first numeric field
else:
    raise ValueError('No numeric column found in the main record set.')
print(f"Using numeric field '@id': {numeric_field}")

# Example: Filter for values above a threshold
threshold = dataframes[main_rs_id][numeric_field].mean()
filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field] > threshold].copy()

print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records\n")
pprint(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"\nNormalized values for {numeric_field}:\n")
pprint(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Pick a string column for grouping (e.g. 'description', species, etc.)
categorical_cols = dataframes[main_rs_id].select_dtypes(include=['object']).columns.tolist()
group_field = None
for col in categorical_cols:
    if col != numeric_field:
        group_field = col
        break
if group_field:
    print(f"\nGrouping by field '@id': {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print("\nGrouped mean:")
    pprint(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and the mean value per group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field], kde=True, bins=30)
plt.title(f"Distribution of {numeric_field} (filtered > mean)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field:
    plt.figure(figsize=(10, 4))
    sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated:
- Loading and exploring a rich mass spectrometry dataset using the `mlcroissant` library.
- Programmatically discovering record sets and their schemas (fields and types by `@id`).
- Extracting the main data table and performing basic exploratory data analysis (EDA), including filtering, normalization, and grouping.
- Visualizing field distributions and group-level means.

This workflow can be adapted for any Croissant-compliant dataset.
